# OCI Robot Cloud — Interactive Quickstart

> **From synthetic data to a fine-tuned GR00T policy in 15 minutes**

This notebook walks you through the full OCI Robot Cloud pipeline interactively:

| Step | What happens | Time |
|------|-------------|------|
| 1 | Synthetic data generation (Genesis SDG) | ~2 min |
| 2 | Dataset inspection (LeRobot v2 format) | ~30 sec |
| 3 | GR00T fine-tuning (command shown; uses pre-built checkpoint) | skipped |
| 4 | Closed-loop evaluation | ~3 min |
| 5 | Results analysis & journey report | ~1 min |

**Prerequisites:** OCI A100 instance with Genesis + Isaac-GR00T venvs installed.  
See [`docs/design_partner_guide.md`](../docs/design_partner_guide.md) for setup instructions.

In [ ]:
# Check environment
import subprocess, sys, os
from pathlib import Path
import json

ok = True

# --- Python version ---
pv = sys.version_info
assert pv >= (3, 9), f"Python 3.9+ required, got {pv.major}.{pv.minor}"
print(f"[OK] Python {pv.major}.{pv.minor}.{pv.micro}")

# --- torch ---
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"[OK] PyTorch {torch.__version__} | CUDA available: {cuda_ok}")
    if cuda_ok:
        print(f"     GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("     [WARN] No GPU detected — GPU cells will be skipped")
except ImportError:
    print("[WARN] torch not installed — GPU cells will be skipped")
    ok = False

# --- genesis ---
try:
    import genesis
    print(f"[OK] genesis {genesis.__version__}")
except ImportError:
    print("[WARN] genesis not installed — run: pip install genesis-world==0.4.3")
    ok = False

# --- pandas + pyarrow (for LeRobot dataset) ---
try:
    import pandas as pd
    import pyarrow
    print(f"[OK] pandas {pd.__version__} | pyarrow {pyarrow.__version__}")
except ImportError as e:
    print(f"[WARN] {e} — run: pip install pandas pyarrow")
    ok = False

# --- project root ---
ROBOTICSAI = Path.home() / "roboticsai"
assert ROBOTICSAI.exists(), f"Expected project root at {ROBOTICSAI}"
print(f"[OK] Project root: {ROBOTICSAI}")

print()
if ok:
    print("Environment looks good — ready to run the pipeline!")
else:
    print("Some dependencies are missing — cells that need them will fail gracefully.")

---

## Step 1: Synthetic Data Generation

We use **Genesis 0.4.3** — a physics simulator that runs at ~38 fps on OCI A100 — to generate robot demonstration episodes via motion-planned IK trajectories.

The script `genesis_sdg_planned.py` runs the Franka Panda arm through a pick-and-lift task:
1. Home → pre-grasp (above red cube)
2. Grasp (close gripper)
3. Lift (above table surface)

Each demo captures joint positions, velocities, gripper state, and RGB images at 256×256.  
With `--num-demos 10` this takes roughly 30 seconds on an A100.

In [ ]:
# NOTE: Skip this cell if running without GPU
%%bash
ROBOTICSAI="$HOME/roboticsai"
OUTPUT_DIR="/tmp/quickstart_sdg"
NUM_DEMOS=10

echo "=== Synthetic Data Generation ==="
echo "Output dir: $OUTPUT_DIR"
echo "Demos:      $NUM_DEMOS"
echo ""

# Activate Genesis venv if available
if [ -d "$HOME/genesis_venv" ]; then
    source "$HOME/genesis_venv/bin/activate"
fi

python3 "$ROBOTICSAI/src/simulation/genesis_sdg_planned.py" \
    --num-demos $NUM_DEMOS \
    --output "$OUTPUT_DIR" \
    --img-size 256

echo ""
echo "=== SDG complete ==="
echo "Files generated:"
ls -lh "$OUTPUT_DIR/" 2>/dev/null | head -20

### What is LeRobot v2 format?

After generation, raw Genesis HDF5 demos are converted to **LeRobot v2** — the standardized dataset format used by NVIDIA's Isaac-GR00T fine-tuning pipeline:

```
/tmp/quickstart_lerobot/
├── meta/
│   ├── info.json          # episode count, fps, action/obs shapes
│   └── episodes.jsonl     # per-episode metadata
├── data/
│   └── chunk-000/
│       ├── observation.state.parquet   # joint positions (9-DOF)
│       ├── action.parquet              # target joint commands
│       └── observation.images.front/  # 256×256 RGB frames
└── videos/                # optional MP4 recordings
```

This format is compatible with `gr00t/experiment/launch_finetune.py` out of the box.

In [ ]:
# NOTE: Skip this cell if running without GPU
%%bash
# Convert Genesis output to LeRobot v2 format
ROBOTICSAI="$HOME/roboticsai"
SDG_DIR="/tmp/quickstart_sdg"
LEROBOT_DIR="/tmp/quickstart_lerobot"

if [ -d "$HOME/Isaac-GR00T/.venv" ]; then
    source "$HOME/Isaac-GR00T/.venv/bin/activate"
fi

echo "=== Converting to LeRobot v2 format ==="
python3 "$ROBOTICSAI/src/training/genesis_to_lerobot.py" \
    --input "$SDG_DIR" \
    --output "$LEROBOT_DIR" \
    --task "pick the red cube from the table" \
    --fps 20

echo ""
echo "Conversion complete. Dataset info:"
cat "$LEROBOT_DIR/meta/info.json" 2>/dev/null || echo "(info.json not found — check output dir)"

---

## Step 2: Dataset Inspection

Before training it is good practice to verify the dataset looks correct.  
The cell below loads the LeRobot Parquet files with pandas and prints episode/frame statistics.

In [ ]:
import json
import pandas as pd
from pathlib import Path

LEROBOT_DIR = Path("/tmp/quickstart_lerobot")

# --- Meta ---
info_path = LEROBOT_DIR / "meta" / "info.json"
if not info_path.exists():
    print(f"[WARN] Dataset not found at {LEROBOT_DIR}")
    print("       Run Step 1 first, or change LEROBOT_DIR to an existing dataset.")
    print("       Showing expected schema instead.")
    print(json.dumps({
        "fps": 20,
        "total_episodes": 10,
        "total_frames": 1000,
        "shapes": {
            "action": [9],
            "observation.state": [9],
            "observation.images.front": [3, 256, 256]
        }
    }, indent=2))
else:
    info = json.loads(info_path.read_text())
    print("=== Dataset info.json ===")
    print(json.dumps(info, indent=2))

    # --- Actions parquet ---
    action_files = sorted((LEROBOT_DIR / "data").rglob("action.parquet"))
    if action_files:
        df_action = pd.concat([pd.read_parquet(f) for f in action_files], ignore_index=True)
        print(f"\n=== Action tensor ===")
        print(f"Rows (frames):  {len(df_action):,}")
        print(f"Columns (DOF):  {len(df_action.columns)}")
        print(f"Action range:   [{df_action.values.min():.3f}, {df_action.values.max():.3f}]")
        print(df_action.describe().round(3))
    
    # --- States parquet ---
    state_files = sorted((LEROBOT_DIR / "data").rglob("observation.state.parquet"))
    if state_files:
        df_state = pd.concat([pd.read_parquet(f) for f in state_files], ignore_index=True)
        print(f"\n=== Observation state ===")
        print(f"Rows (frames):  {len(df_state):,}")
        print(f"Columns (DOF):  {len(df_state.columns)}")

    # --- Episode count from jsonl ---
    episodes_path = LEROBOT_DIR / "meta" / "episodes.jsonl"
    if episodes_path.exists():
        episodes = [json.loads(l) for l in episodes_path.read_text().strip().splitlines()]
        lengths = [ep.get("length", 0) for ep in episodes]
        print(f"\n=== Episodes ===")
        print(f"Count:         {len(episodes)}")
        print(f"Avg length:    {sum(lengths)/len(lengths):.1f} frames")
        print(f"Min length:    {min(lengths)} frames")
        print(f"Max length:    {max(lengths)} frames")

---

## Step 3: GR00T Fine-Tuning

Fine-tuning GR00T N1.6 (3B parameters) on the synthetic dataset uses `launch_finetune.py` from the Isaac-GR00T repo.  
A full 1000-demo × 5000-step run takes ~35 minutes on a single OCI A100 at **$0.0043 / 10k steps**.

**The cell below shows the exact command — actual training is skipped in this quickstart notebook.**  
We use a pre-existing checkpoint (`/tmp/finetune_1000_5k/checkpoint-5000`) for evaluation.

In [ ]:
# NOTE: Skip this cell if running without GPU
# This cell prints the fine-tuning command — it does NOT run training.
# To run training, copy the command to a terminal and execute it.

from pathlib import Path

ROBOTICSAI = Path.home() / "roboticsai"
MODEL_PATH = "/home/ubuntu/models/GR00T-N1.6-3B"
DATASET_PATH = "/tmp/quickstart_lerobot"
OUTPUT_DIR = "/tmp/quickstart_finetune"
MAX_STEPS = 500  # quickstart: 500; production: 5000
GPU = 4

cmd = f"""\
cd ~/Isaac-GR00T && source .venv/bin/activate

CUDA_VISIBLE_DEVICES={GPU} python gr00t/experiment/launch_finetune.py \\
    --base-model-path {MODEL_PATH} \\
    --dataset-path {DATASET_PATH} \\
    --embodiment-tag NEW_EMBODIMENT \\
    --modality-config-path {ROBOTICSAI}/src/training/franka_config.py \\
    --num-gpus 1 \\
    --output-dir {OUTPUT_DIR} \\
    --save-total-limit 3 \\
    --save-steps {MAX_STEPS} \\
    --max-steps {MAX_STEPS} \\
    --global-batch-size 16 \\
    --dataloader-num-workers 4
"""

print("=== GR00T Fine-Tuning Command ===")
print(cmd)
print()
print(f"Estimated time (500 steps, 10 demos):  ~3 min on A100")
print(f"Estimated time (5000 steps, 1000 demos): ~35 min on A100")
print(f"Cost (5000 steps, OCI A100):           ~$0.002")
print()
print("For this quickstart, we will use the pre-existing checkpoint:")
print("  /tmp/finetune_1000_5k/checkpoint-5000")

---

## Step 4: Evaluate

We start the GR00T Franka inference server (`groot_franka_server.py`) in the background on port 8002,  
then run `closed_loop_eval.py` for 10 episodes.

The closed-loop eval uses a live Genesis simulation: at each step the robot's current joint state + camera image  
are sent to the GR00T server, which returns 16-step action chunks (arm × 7 DOF + gripper × 2).

In [ ]:
# NOTE: Skip this cell if running without GPU
%%bash
ROBOTICSAI="$HOME/roboticsai"
CHECKPOINT="/tmp/finetune_1000_5k/checkpoint-5000"
EVAL_OUTPUT="/tmp/quickstart_eval"
PORT=8002

# Check checkpoint exists; fall back to mock mode
if [ ! -d "$CHECKPOINT" ]; then
    echo "[WARN] Checkpoint not found at $CHECKPOINT"
    echo "       Running in mock mode (no real GR00T inference)"
    MOCK_FLAG="--mock"
else
    MOCK_FLAG=""
    echo "Using checkpoint: $CHECKPOINT"

    # Start GR00T Franka server in background
    echo "Starting GR00T server on port $PORT..."
    if [ -d "$HOME/Isaac-GR00T/.venv" ]; then
        source "$HOME/Isaac-GR00T/.venv/bin/activate"
    fi
    python3 "$ROBOTICSAI/src/inference/groot_franka_server.py" \
        --checkpoint "$CHECKPOINT" \
        --port $PORT &
    SERVER_PID=$!
    echo "Server PID: $SERVER_PID"

    # Wait for server to be ready
    echo "Waiting for server to become healthy..."
    for i in $(seq 1 30); do
        if curl -sf "http://localhost:$PORT/health" > /dev/null 2>&1; then
            echo "Server ready after ${i}s"
            break
        fi
        sleep 1
    done
fi

echo ""
echo "=== Running closed-loop evaluation ==="
if [ -d "$HOME/genesis_venv" ]; then
    source "$HOME/genesis_venv/bin/activate"
fi

python3 "$ROBOTICSAI/src/eval/closed_loop_eval.py" \
    --server-url "http://localhost:$PORT" \
    --num-episodes 10 \
    --output-dir "$EVAL_OUTPUT" \
    $MOCK_FLAG

echo ""
echo "=== Evaluation complete ==="
cat "$EVAL_OUTPUT/summary.json" 2>/dev/null || echo "(summary.json not found)"

# Stop server if we started it
if [ ! -z "${SERVER_PID:-}" ]; then
    kill $SERVER_PID 2>/dev/null && echo "Server stopped."
fi

In [ ]:
# Show eval results as formatted JSON
import json
from pathlib import Path

EVAL_OUTPUT = Path("/tmp/quickstart_eval")
summary_path = EVAL_OUTPUT / "summary.json"

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("=== Closed-Loop Evaluation Results ===")
    print(json.dumps(summary, indent=2))

    sr = summary.get("success_rate", 0)
    n = summary.get("num_episodes", 10)
    print(f"\nSuccess rate: {sr:.1%} over {n} episodes")
    if sr == 0.0:
        print("Note: 0% is the expected baseline for a pure open-loop policy.")
        print("      Run DAgger (dagger_run5.sh) to push beyond 5-65% collection success.")
else:
    print("[WARN] Eval output not found. Showing expected summary schema:")
    print(json.dumps({
        "success_rate": 0.05,
        "num_episodes": 10,
        "avg_latency_ms": 231.4,
        "checkpoint": "/tmp/finetune_1000_5k/checkpoint-5000",
        "timestamp": "2026-03-29T00:00:00"
    }, indent=2))

---

## Step 5: Results Analysis

The `generate_journey_report.py` script produces a self-contained dark-theme HTML page showing:
- Success rate progression (BC baseline → DAgger iterations → 1000-demo run)
- Expert intervention decline chart
- Cost comparison table (OCI vs DGX vs AWS p4d vs Lambda)
- Pipeline architecture diagram

We generate the report below and display it inline.

In [ ]:
# NOTE: Skip this cell if running without GPU
%%bash
ROBOTICSAI="$HOME/roboticsai"
REPORT_PATH="/tmp/quickstart_journey_report.html"

python3 "$ROBOTICSAI/src/eval/generate_journey_report.py" \
    --output "$REPORT_PATH" \
    --eval-dirs /tmp/quickstart_eval

echo "Report generated: $REPORT_PATH"
ls -lh "$REPORT_PATH" 2>/dev/null

In [ ]:
# Display the journey report inline in the notebook
from pathlib import Path
from IPython.display import HTML, display

REPORT_PATH = Path("/tmp/quickstart_journey_report.html")

if REPORT_PATH.exists():
    html_content = REPORT_PATH.read_text()
    # Wrap in an iframe-like scrollable div to avoid full-page style bleed
    display(HTML(f"""
    <div style="width:100%; height:600px; overflow:auto; border:1px solid #333; border-radius:8px;">
        {html_content}
    </div>
    """))
else:
    print("[WARN] Report not found at", REPORT_PATH)
    print("Run the bash cell above to generate it first.")
    print()
    print("If you want to view an existing report, change REPORT_PATH to one of:")
    print("  /tmp/journey_report.html  (if you ran generate_journey_report.py manually)")

---

## Cost Estimate

In [ ]:
# OCI Robot Cloud — pipeline cost estimate
# Based on OCI A100 (BM.GPU.A100-v2.8), priced per GPU-hour

# --- Config ---
NUM_DEMOS        = 1000        # SDG demos
TRAIN_STEPS      = 5000        # fine-tuning steps
EVAL_EPISODES    = 20          # closed-loop eval episodes

# --- OCI pricing (per GPU-hour, single A100) ---
OCI_A100_PER_HR  = 2.95        # USD/hr (on-demand)

# --- Timing from benchmarks ---
SDG_FPS          = 38.5        # Genesis on A100 (session5)
STEPS_PER_DEMO   = 100         # frames per demo
FINETUNE_ITPS    = 2.35        # it/s (session5 benchmark)
EVAL_LATENCY_S   = 0.231       # GR00T server avg latency
EVAL_STEPS       = 100         # sim steps per episode

# --- Calculations ---
sdg_frames  = NUM_DEMOS * STEPS_PER_DEMO
sdg_sec     = sdg_frames / SDG_FPS
train_sec   = TRAIN_STEPS / FINETUNE_ITPS
eval_sec    = EVAL_EPISODES * EVAL_STEPS * EVAL_LATENCY_S
total_sec   = sdg_sec + train_sec + eval_sec
total_hr    = total_sec / 3600
total_cost  = total_hr * OCI_A100_PER_HR

cost_per_10k_steps = (OCI_A100_PER_HR / 3600) * (10000 / FINETUNE_ITPS)

print("=" * 52)
print(" OCI Robot Cloud — Pipeline Cost Estimate")
print("=" * 52)
print(f" Config: {NUM_DEMOS} demos × {TRAIN_STEPS} steps × {EVAL_EPISODES} eval eps")
print()
print(f" SDG generation:    {sdg_sec/60:5.1f} min")
print(f" Fine-tuning:       {train_sec/60:5.1f} min")
print(f" Closed-loop eval:  {eval_sec/60:5.1f} min")
print(f" Total wall time:   {total_sec/60:5.1f} min  ({total_hr:.2f} GPU-hr)")
print()
print(f" OCI A100 rate:     ${OCI_A100_PER_HR}/GPU-hr")
print(f" Total cost:        ${total_cost:.4f}")
print(f" Cost per 10k steps:${cost_per_10k_steps:.4f}")
print()
print(" For comparison (benchmark data from cloud_benchmark.py):")
print(f"   AWS p4d (A100):  ~${cost_per_10k_steps * 9.6:.4f} / 10k steps  (9.6× more expensive)")
print(f"   Lambda Cloud:    ~${cost_per_10k_steps * 4.2:.4f} / 10k steps  (4.2× more expensive)")
print("=" * 52)

---

## Next Steps

You have completed the full OCI Robot Cloud quickstart: **SDG → dataset inspection → fine-tune → closed-loop eval → results report**.

### Improve success rate with DAgger

The baseline open-loop policy achieves ~5% closed-loop success. Run iterative data aggregation to push toward 65%+:

```bash
# DAgger run5: 5 iters × 20 episodes, starting from 1000-demo checkpoint
bash ~/roboticsai/src/training/dagger_run5.sh
```

### Multi-GPU fine-tuning (4× A100)

```bash
bash ~/roboticsai/src/training/finetune_multigpu.sh
# 3.07× throughput, 230 samples/sec
```

### Design partner onboarding

- Full onboarding guide: [`docs/design_partner_guide.md`](../docs/design_partner_guide.md)
- Upload your own robot demos via the data collection API (port 8003)
- Watch training live via the training monitor dashboard (port 8004)

### Contact

For access, questions, or partnership inquiries, reach out via the OCI Robot Cloud design-partner program.  
See [`docs/design_partner_guide.md`](../docs/design_partner_guide.md) for contact details.

---
*Notebook version: 2026-03-29 | OCI Robot Cloud v0.1 | Compatible with OCI JupyterLab*